In [ ]:
# Chest X-Ray Pneumonia - Multi-Scale Frequency Pyramid Fusion CNN
# Key Innovation: Hierarchical Frequency-Spatial Attention (HFSA) before GAP
# Dataset: Chest X-Ray Images (Pneumonia)
#   https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
# Classes: NORMAL, PNEUMONIA (binary classification)
#
# ADAPTED FROM: Brain Tumor MRI Improved Multi-Scale Frequency Pyramid Fusion CNN
# KEY CHANGES:
#   1. Binary classification: NORMAL vs PNEUMONIA (2 classes)
#   2. Dataset path updated for chest_xray structure (train/val/test)
#   3. Class imbalance handling: WeightedRandomSampler + weighted CE loss
#   4. Additional augmentations suited for X-ray imaging
#   5. Comprehensive metrics: Accuracy, Precision, Recall, F1, Cohen's Kappa,
#      Specificity (all reported as overall / macro / per-class)
#   6. Val split: dataset already has a val folder — used directly

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from tqdm.auto import tqdm
import os
import warnings
import gc
from collections import Counter
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score,
    f1_score, cohen_kappa_score
)
import seaborn as sns
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


# ================== Step 1: Dataset ==================

class ChestXRayDataset(Dataset):
    """
    Loads the Chest X-Ray Pneumonia dataset.
    Expected structure:
        root/train/NORMAL/   root/train/PNEUMONIA/
        root/val/NORMAL/     root/val/PNEUMONIA/
        root/test/NORMAL/    root/test/PNEUMONIA/
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir)
                                if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.samples.append((
                            os.path.join(class_dir, img_name),
                            self.class_to_idx[class_name]
                        ))
        label_counts = Counter([s[1] for s in self.samples])
        print(f"Found {len(self.samples)} images in {len(self.classes)} classes: {self.classes}")
        for cls, idx in self.class_to_idx.items():
            print(f"  {cls}: {label_counts[idx]} images")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            # X-rays are often grayscale; convert to RGB for ResNet
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros(3, 224, 224), label

    def get_class_weights(self):
        """Returns inverse-frequency weights for WeightedRandomSampler."""
        label_counts = Counter([s[1] for s in self.samples])
        total = len(self.samples)
        weights = []
        for _, label in self.samples:
            weights.append(total / (len(label_counts) * label_counts[label]))
        return torch.FloatTensor(weights)


def load_chest_xray_dataset(data_root):
    # X-ray-specific augmentation: no color jitter on saturation (greyscale),
    # slight CLAHE-like brightness/contrast boost, conservative flips
    transform_train = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.3, contrast=0.3),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    transform_eval = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dir = os.path.join(data_root, 'train')
    val_dir   = os.path.join(data_root, 'val')
    test_dir  = os.path.join(data_root, 'test')

    trainset = ChestXRayDataset(train_dir, transform=transform_train)
    valset   = ChestXRayDataset(val_dir,   transform=transform_eval)
    testset  = ChestXRayDataset(test_dir,  transform=transform_eval)

    return trainset, valset, testset, trainset.classes


# ================== Step 2: FFT Conversion Functions ==================

def spatial_to_frequency(images):
    images = images.float()
    freq_complex = fft.fft2(images, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    freq_magnitude = torch.abs(freq_complex)
    freq_phase = torch.angle(freq_complex)
    eps = 1e-6
    freq_magnitude = torch.clamp(freq_magnitude, min=eps)
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    mean_val = freq_magnitude_log.mean()
    std_val = torch.clamp(freq_magnitude_log.std(), min=1e-5)
    freq_magnitude_normalized = (freq_magnitude_log - mean_val) / std_val
    freq_magnitude_normalized = torch.clamp(freq_magnitude_normalized, -10, 10)
    phase_cos = torch.cos(freq_phase)
    phase_sin = torch.sin(freq_phase)
    freq_features = torch.cat([freq_magnitude_normalized, phase_cos, phase_sin], dim=1)
    freq_features = torch.nan_to_num(freq_features, nan=0.0, posinf=10.0, neginf=-10.0)
    return freq_features, freq_phase, freq_complex


def frequency_to_spatial(freq_magnitude, freq_phase):
    freq_magnitude = torch.exp(torch.clamp(freq_magnitude, -10, 10))
    freq_complex = freq_magnitude * torch.exp(1j * freq_phase)
    freq_complex = fft.ifftshift(freq_complex, dim=(-2, -1))
    spatial_complex = fft.ifft2(freq_complex, dim=(-2, -1))
    spatial_images = torch.real(spatial_complex)
    return spatial_images


def extract_frequency_features(activation_map):
    """FFT on activation maps — returns same-shape frequency magnitude."""
    activation_map = activation_map.float()
    freq_complex = fft.fft2(activation_map, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    freq_magnitude = torch.abs(freq_complex)
    eps = 1e-6
    freq_magnitude = torch.clamp(freq_magnitude, min=eps)
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    b, c, h, w = freq_magnitude_log.shape
    freq_magnitude_norm = freq_magnitude_log.view(b, c, -1)
    mean_val = freq_magnitude_norm.mean(dim=2, keepdim=True)
    std_val  = torch.clamp(freq_magnitude_norm.std(dim=2, keepdim=True), min=1e-5)
    freq_magnitude_norm = (freq_magnitude_norm - mean_val) / std_val
    freq_magnitude_norm = torch.clamp(freq_magnitude_norm, -10, 10)
    freq_magnitude_norm = freq_magnitude_norm.view(b, c, h, w)
    freq_magnitude_norm = torch.nan_to_num(freq_magnitude_norm, nan=0.0, posinf=10.0, neginf=-10.0)
    return freq_magnitude_norm


# ================== Step 3: Frequency Domain Dataset ==================

class FrequencyDomainDataset(Dataset):
    def __init__(self, original_dataset, cache_freq=False):
        self.original_dataset = original_dataset
        self.cache_freq = cache_freq
        self.freq_cache = {} if cache_freq else None

    def __len__(self):
        return len(self.original_dataset)

    def __getitem__(self, idx):
        if self.cache_freq and idx in self.freq_cache:
            return self.freq_cache[idx]
        image, label = self.original_dataset[idx]
        with torch.no_grad():
            freq_features, freq_phase, _ = spatial_to_frequency(image.unsqueeze(0))
            freq_features = freq_features.squeeze(0)
            freq_phase    = freq_phase.squeeze(0)
        result = (freq_features, label, freq_phase)
        if self.cache_freq:
            self.freq_cache[idx] = result
        return result


# ================== Step 4: Model Architecture ==================

class DualDomainFeatureFusion(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.spatial_weight = nn.Parameter(torch.tensor(0.7))
        self.freq_weight    = nn.Parameter(torch.tensor(0.3))
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels * 2, kernel_size=1),
            nn.Sigmoid()
        )
        self.bn_spatial = nn.BatchNorm2d(in_channels)
        self.bn_freq    = nn.BatchNorm2d(in_channels)

    def forward(self, spatial_features):
        freq_features         = extract_frequency_features(spatial_features)
        spatial_features_norm = self.bn_spatial(spatial_features)
        freq_features_norm    = self.bn_freq(freq_features)
        combined              = torch.cat([spatial_features_norm, freq_features_norm], dim=1)
        attention             = self.channel_attention(combined)
        combined_weighted     = combined * attention
        spatial_weighted      = combined_weighted[:, :spatial_features.size(1), :, :]
        freq_weighted         = combined_weighted[:, spatial_features.size(1):, :, :]
        spatial_w             = torch.sigmoid(self.spatial_weight)
        freq_w                = torch.sigmoid(self.freq_weight)
        fused                 = spatial_w * spatial_weighted + freq_w * freq_weighted
        fused                 = torch.nan_to_num(fused, nan=0.0, posinf=10.0, neginf=-10.0)
        return fused


class FrequencyAttentionGate(nn.Module):
    def __init__(self, in_channels, reduction=8):
        super().__init__()
        mid = max(in_channels // reduction, 32)
        self.freq_gate = nn.Sequential(
            nn.Conv2d(in_channels, mid, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, in_channels, kernel_size=1, bias=False),
            nn.Sigmoid()
        )
        self.bn    = nn.BatchNorm2d(in_channels)
        self.alpha = nn.Parameter(torch.tensor(0.2))

    def forward(self, x):
        freq_mag = extract_frequency_features(x)
        gate     = self.freq_gate(self.bn(freq_mag))
        alpha    = torch.sigmoid(self.alpha)
        out      = x * (1.0 + alpha * gate)
        out      = torch.nan_to_num(out, nan=0.0, posinf=10.0, neginf=-10.0)
        return out


class HierarchicalFPN(nn.Module):
    def __init__(self, channels=[256, 512, 1024, 2048]):
        super().__init__()
        C1, C2, C3, C4 = channels
        self.freq_gate2  = FrequencyAttentionGate(C2, reduction=8)
        self.freq_gate3  = FrequencyAttentionGate(C3, reduction=8)
        self.dual_domain4 = DualDomainFeatureFusion(C4)
        self.lat3 = nn.Sequential(
            nn.Conv2d(C3, C4, kernel_size=1, bias=False),
            nn.BatchNorm2d(C4), nn.ReLU(inplace=True)
        )
        self.lat2 = nn.Sequential(
            nn.Conv2d(C2, C4, kernel_size=1, bias=False),
            nn.BatchNorm2d(C4), nn.ReLU(inplace=True)
        )
        self.scale_logits = nn.Parameter(torch.zeros(3))
        self.merge_conv   = nn.Sequential(
            nn.Conv2d(C4, C4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(C4), nn.ReLU(inplace=True)
        )

    def forward(self, x1, x2, x3, x4):
        x2_f = self.freq_gate2(x2)
        x3_f = self.freq_gate3(x3)
        x4_f = self.dual_domain4(x4)
        p4   = x4_f
        p3   = F.interpolate(p4, size=x3_f.shape[-2:], mode='nearest') + self.lat3(x3_f)
        p2   = F.interpolate(p3, size=x2_f.shape[-2:], mode='nearest') + self.lat2(x2_f)
        p2_s = F.adaptive_avg_pool2d(p2, (7, 7))
        p3_s = F.adaptive_avg_pool2d(p3, (7, 7))
        p4_s = p4
        w    = F.softmax(self.scale_logits, dim=0)
        merged = w[0] * p2_s + w[1] * p3_s + w[2] * p4_s
        out  = self.merge_conv(merged)
        out  = torch.nan_to_num(out, nan=0.0, posinf=10.0, neginf=-10.0)
        return out


class ChestXRayFrequencyCNN(nn.Module):
    """
    ResNet50 + Hierarchical FPN with per-scale Frequency Attention Gates.
    Adapted for binary chest X-ray classification (NORMAL vs PNEUMONIA).
    """
    def __init__(self, num_classes=2, dropout_rate=0.5):
        super().__init__()
        self.backbone = models.resnet50(pretrained=True)

        # Replace first conv for 9-channel frequency input
        original_conv1 = self.backbone.conv1
        self.backbone.conv1 = nn.Conv2d(9, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            self.backbone.conv1.weight[:, :3, :, :] = original_conv1.weight
            nn.init.kaiming_normal_(self.backbone.conv1.weight[:, 3:, :, :],
                                    mode='fan_out', nonlinearity='relu')
            self.backbone.conv1.weight[:, 3:, :, :] *= 0.1

        self.hfpn       = HierarchicalFPN(channels=[256, 512, 1024, 2048])
        self.gap        = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(256, num_classes)
        )
        self._initialize_new_weights()

    def _initialize_new_weights(self):
        for m in self.hfpn.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)

    def _backbone_features(self, x):
        x  = torch.nan_to_num(x, nan=0.0, posinf=10.0, neginf=-10.0)
        x  = self.backbone.conv1(x)
        x  = self.backbone.bn1(x)
        x  = self.backbone.relu(x)
        x  = self.backbone.maxpool(x)
        x1 = self.backbone.layer1(x)
        x2 = self.backbone.layer2(x1)
        x3 = self.backbone.layer3(x2)
        x4 = self.backbone.layer4(x3)
        return x1, x2, x3, x4

    def forward(self, x):
        x1, x2, x3, x4 = self._backbone_features(x)
        fused  = self.hfpn(x1, x2, x3, x4)
        pooled = self.gap(fused)
        pooled = torch.flatten(pooled, 1)
        return self.classifier(pooled)

    def get_activations(self, x):
        _, _, _, x4 = self._backbone_features(x)
        return x4

    def get_fused_activations(self, x):
        x1, x2, x3, x4 = self._backbone_features(x)
        return self.hfpn(x1, x2, x3, x4)


# ================== Step 5: Metrics ==================

def compute_all_metrics(all_labels, all_predictions, classes):
    """
    Computes and prints a comprehensive metrics table:
      - Overall Accuracy
      - Overall Precision (macro)
      - Overall Recall / Sensitivity (macro)
      - Overall F1 Score (macro)
      - Overall Cohen's Kappa
      - Overall Specificity (macro)
      - Per-class versions of all the above
    Returns a dict of metric values.
    """
    all_labels      = np.array(all_labels)
    all_predictions = np.array(all_predictions)
    num_classes     = len(classes)

    # --- Overall scalars ---
    overall_accuracy  = accuracy_score(all_labels, all_predictions)
    overall_precision = precision_score(all_labels, all_predictions, average='macro', zero_division=0)
    overall_recall    = recall_score(all_labels, all_predictions, average='macro', zero_division=0)
    overall_f1        = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
    overall_kappa     = cohen_kappa_score(all_labels, all_predictions)

    # Specificity: for each class c, spec_c = TN_c / (TN_c + FP_c)
    cm                 = confusion_matrix(all_labels, all_predictions)
    per_class_spec     = []
    per_class_sens     = []
    per_class_prec     = []
    per_class_f1       = []
    for c in range(num_classes):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FP - FN
        spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        sens = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        prec = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        f1c  = (2 * prec * sens / (prec + sens)) if (prec + sens) > 0 else 0.0
        per_class_spec.append(spec)
        per_class_sens.append(sens)
        per_class_prec.append(prec)
        per_class_f1.append(f1c)

    overall_specificity = np.mean(per_class_spec)

    # --- Print table ---
    sep = "=" * 70
    print(f"\n{sep}")
    print("  COMPREHENSIVE METRICS SUMMARY")
    print(sep)
    print(f"  {'Metric':<30} {'Overall (Macro)':>20}")
    print("-" * 55)
    print(f"  {'Accuracy':<30} {overall_accuracy*100:>19.4f}%")
    print(f"  {'Precision (macro)':<30} {overall_precision*100:>19.4f}%")
    print(f"  {'Recall / Sensitivity (macro)':<30} {overall_recall*100:>19.4f}%")
    print(f"  {'F1 Score (macro)':<30} {overall_f1*100:>19.4f}%")
    print(f"  {'Specificity (macro)':<30} {overall_specificity*100:>19.4f}%")
    print(f"  {'Cohen\'s Kappa':<30} {overall_kappa:>20.4f}")
    print(sep)
    print("\n  PER-CLASS METRICS")
    print("-" * 70)
    header = f"  {'Class':<16} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Specificity':>12}"
    print(header)
    print("-" * 70)
    for c, cls_name in enumerate(classes):
        print(f"  {cls_name:<16} "
              f"{per_class_prec[c]*100:>9.4f}% "
              f"{per_class_sens[c]*100:>9.4f}% "
              f"{per_class_f1[c]*100:>9.4f}% "
              f"{per_class_spec[c]*100:>11.4f}%")
    print(sep)

    # Also print sklearn's full report for reference
    print("\n  SKLEARN CLASSIFICATION REPORT")
    print("-" * 70)
    print(classification_report(all_labels, all_predictions,
                                 target_names=classes, digits=4))
    print(sep)

    return {
        'accuracy':    overall_accuracy,
        'precision':   overall_precision,
        'recall':      overall_recall,
        'f1':          overall_f1,
        'kappa':       overall_kappa,
        'specificity': overall_specificity,
        'per_class_precision':    per_class_prec,
        'per_class_recall':       per_class_sens,
        'per_class_f1':           per_class_f1,
        'per_class_specificity':  per_class_spec,
        'confusion_matrix':       cm
    }


# ================== Step 6: Training ==================

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0, verbose=True):
        self.patience        = patience
        self.min_delta       = min_delta
        self.verbose         = verbose
        self.counter         = 0
        self.best_score      = None
        self.early_stop      = False
        self.best_model_state = None

    def __call__(self, val_accuracy, model):
        score = val_accuracy
        if self.best_score is None:
            self.best_score      = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score      = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter         = 0


def _is_new_param(name):
    new_prefixes = ('backbone.conv1', 'hfpn.', 'classifier.')
    return any(name.startswith(p) for p in new_prefixes)


def train_model(model, train_loader, val_loader, class_weights,
                epochs=50, lr=0.001, weight_decay=1e-4):
    # Weighted cross-entropy to handle class imbalance
    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device),
        label_smoothing=0.05        # mild smoothing for binary task
    )

    pretrained_params, new_params = [], []
    for name, param in model.named_parameters():
        (new_params if _is_new_param(name) else pretrained_params).append(param)

    optimizer = torch.optim.AdamW([
        {'params': pretrained_params, 'lr': lr * 0.01},
        {'params': new_params,        'lr': lr * 0.5}
    ], weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=15, T_mult=2, eta_min=1e-7
    )

    early_stopping = EarlyStopping(patience=15, min_delta=0.1, verbose=True)
    train_losses, val_losses, train_accuracies, val_accuracies = [], [], [], []
    best_val_accuracy = 0.0
    best_model_state  = None

    scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

    for epoch in range(epochs):
        # ---- Training ----
        model.train()
        running_loss   = 0.0
        correct_train  = 0
        total_train    = 0

        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for i, (freq_images, labels, _) in enumerate(train_pbar):
            freq_images = freq_images.to(device, non_blocking=True)
            labels      = labels.to(device, non_blocking=True)

            if torch.isnan(freq_images).any() or torch.isinf(freq_images).any():
                print(f"Warning: NaN/Inf in input batch {i}, skipping...")
                continue

            optimizer.zero_grad(set_to_none=True)

            if scaler is not None:
                with torch.cuda.amp.autocast():
                    outputs = model(freq_images)
                    loss    = criterion(outputs, labels)
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss at batch {i}, skipping...")
                    continue
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(freq_images)
                loss    = criterion(outputs, labels)
                if torch.isnan(loss) or torch.isinf(loss):
                    continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            running_loss   += loss.item()
            _, predicted    = torch.max(outputs.data, 1)
            total_train    += labels.size(0)
            correct_train  += (predicted == labels).sum().item()
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc':  f'{100 * correct_train / total_train:.2f}%'
            })
            if i % 50 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()

        scheduler.step()
        avg_train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct_train / total_train
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)

        # ---- Validation ----
        model.eval()
        running_val_loss = 0.0
        correct = 0; total = 0

        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
            for freq_images, labels, _ in val_pbar:
                freq_images = freq_images.to(device, non_blocking=True)
                labels      = labels.to(device, non_blocking=True)
                if scaler is not None:
                    with torch.cuda.amp.autocast():
                        outputs = model(freq_images)
                        loss    = criterion(outputs, labels)
                else:
                    outputs = model(freq_images)
                    loss    = criterion(outputs, labels)
                running_val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total   += labels.size(0)
                correct += (predicted == labels).sum().item()
                val_pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc':  f'{100 * correct / total:.2f}%'
                })

        avg_val_loss  = running_val_loss / len(val_loader)
        val_accuracy  = 100 * correct / total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(f'\nEpoch [{epoch+1}/{epochs}]')
        print(f'  Train Loss: {avg_train_loss:.4f}  |  Train Acc: {train_accuracy:.2f}%')
        print(f'  Val   Loss: {avg_val_loss:.4f}  |  Val   Acc: {val_accuracy:.2f}%')
        print(f'  LR (backbone): {optimizer.param_groups[0]["lr"]:.6f}')

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model — val accuracy: {best_val_accuracy:.2f}%")

    return train_losses, val_losses, train_accuracies, val_accuracies


# ================== Step 7: Score-CAM ==================

class ScoreCAM:
    def __init__(self, model):
        self.model = model
        self.model.eval()

    def generate_cam(self, input_image, target_class, batch_size=16):
        activations = self.model.get_fused_activations(input_image)
        b, k, h, w  = activations.shape
        _, _, input_h, input_w = input_image.shape
        upsampled = F.interpolate(
            activations, size=(input_h, input_w), mode='bilinear', align_corners=False
        ).squeeze(0)

        weights = []
        for i in range(0, k, batch_size):
            batch_end = min(i + batch_size, k)
            for act_map in upsampled[i:batch_end]:
                act_norm = act_map - act_map.min()
                if act_norm.max() > 0:
                    act_norm = act_norm / act_norm.max()
                masked = input_image * act_norm.unsqueeze(0).unsqueeze(0)
                with torch.no_grad():
                    out   = self.model(masked)
                    score = F.softmax(out, dim=1)[0, target_class].item()
                weights.append(score)

        weights = torch.FloatTensor(weights).to(device)
        if weights.max() > 0:
            weights = weights / weights.max()

        activations_2d = activations.squeeze(0)
        cam = torch.zeros((h, w), dtype=torch.float32).to(device)
        for i, w_val in enumerate(weights):
            cam += w_val * activations_2d[i]
        cam = F.relu(cam)
        if cam.max() > 0:
            cam = cam / cam.max()
        cam = F.interpolate(
            cam.unsqueeze(0).unsqueeze(0),
            size=(input_h, input_w), mode='bilinear', align_corners=False
        ).squeeze()
        return cam.cpu().detach().numpy(), weights.cpu().detach().numpy()


# ================== Step 8: Spatial Domain Mapping ==================

def apply_scorecam_and_map_to_spatial(model, freq_image, phase, target_class, original_image):
    model.eval()
    freq_input = freq_image.clone().detach().to(device)
    scorecam   = ScoreCAM(model)
    cam_freq, weights = scorecam.generate_cam(freq_input, target_class, batch_size=32)
    freq_magnitude    = freq_input[:, :3, :, :].squeeze(0).cpu().detach()
    cam_freq_tensor   = torch.from_numpy(cam_freq).float()
    masked_freq_mag   = freq_magnitude * cam_freq_tensor.unsqueeze(0)
    if phase.dim() == 4:
        phase = phase.squeeze(0)
    elif phase.dim() == 2:
        phase = phase.unsqueeze(0).repeat(3, 1, 1)
    cam_spatial = frequency_to_spatial(
        masked_freq_mag.unsqueeze(0), phase.unsqueeze(0)
    ).squeeze(0)
    cam_spatial  = torch.abs(cam_spatial)
    saliency_map = torch.mean(cam_spatial, dim=0).numpy()
    saliency_map = np.max(saliency_map) - saliency_map
    saliency_map = gaussian_filter(saliency_map, sigma=2.5)
    if saliency_map.max() > saliency_map.min():
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    else:
        saliency_map = np.zeros_like(saliency_map)
    threshold    = np.percentile(saliency_map, 40)
    saliency_map = np.where(saliency_map > threshold, saliency_map, 0)
    from scipy.ndimage import binary_closing, binary_opening
    binary_mask  = saliency_map > 0
    binary_mask  = binary_closing(binary_mask, structure=np.ones((5, 5)))
    binary_mask  = binary_opening(binary_mask, structure=np.ones((3, 3)))
    saliency_map = saliency_map * binary_mask
    if saliency_map.max() > 0:
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    saliency_map = np.power(saliency_map, 0.7)
    if original_image.dim() == 4:
        original_image = original_image.squeeze(0)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    original_denorm = torch.clamp(original_image.cpu() * std + mean, 0, 1)
    original_np     = original_denorm.permute(1, 2, 0).numpy()
    saliency_colored = plt.cm.jet(saliency_map)[:, :, :3]
    alpha            = 0.7 * saliency_map[:, :, np.newaxis]
    highlighted      = np.clip((1 - alpha) * original_np + alpha * saliency_colored, 0, 1)
    return cam_spatial, saliency_map, highlighted, original_np, cam_freq


# ================== Step 9: Visualization ==================

def plot_scorecam_results(original_np, freq_magnitude, cam_freq, saliency_map,
                          highlighted, prediction, true_label, classes, confidence):
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(
        'Chest X-Ray Pneumonia — Hierarchical Frequency-Spatial Attention CNN\n'
        '(FreqAttentionGate @ layer2/3 + DualDomainFusion @ layer4 + FPN merge)',
        fontsize=14, fontweight='bold', y=0.998
    )
    axes[0, 0].imshow(original_np, cmap='gray' if original_np.mean(axis=2).std() < 0.05 else None)
    axes[0, 0].set_title(f'Original X-Ray\nGround Truth: {classes[true_label]}',
                         fontsize=11, fontweight='bold')
    axes[0, 0].axis('off')

    freq_display = freq_magnitude[:, :3, :, :].squeeze(0).mean(0).cpu().numpy()
    im1 = axes[0, 1].imshow(freq_display, cmap='viridis')
    axes[0, 1].set_title('Frequency Domain\n(Input Magnitude Spectrum)', fontsize=11, fontweight='bold')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)

    im2 = axes[0, 2].imshow(cam_freq, cmap='jet')
    axes[0, 2].set_title('Score-CAM\n(HFPN Features)', fontsize=11, fontweight='bold')
    axes[0, 2].axis('off')
    plt.colorbar(im2, ax=axes[0, 2], fraction=0.046, pad=0.04)

    correct = "\u2713" if prediction == true_label else "\u2717"
    color   = 'green' if prediction == true_label else 'red'
    axes[0, 3].text(0.5, 0.5,
                    f'{correct} Prediction:\n{classes[prediction]}\n\nConfidence:\n{confidence:.1f}%',
                    ha='center', va='center', fontsize=13, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
    axes[0, 3].set_title('Model Prediction', fontsize=11, fontweight='bold')
    axes[0, 3].axis('off')

    im3 = axes[1, 0].imshow(saliency_map, cmap='hot')
    axes[1, 0].set_title('Saliency Map\n(HFPN Enhanced)', fontsize=11, fontweight='bold')
    axes[1, 0].axis('off')
    plt.colorbar(im3, ax=axes[1, 0], fraction=0.046, pad=0.04)

    axes[1, 1].imshow(highlighted)
    axes[1, 1].set_title('Highlighted Lung Region\n(Multi-Scale Freq-Aware)', fontsize=11, fontweight='bold')
    axes[1, 1].axis('off')

    axes[1, 2].imshow(original_np)
    axes[1, 2].imshow(saliency_map, cmap='jet', alpha=0.5)
    axes[1, 2].set_title('Importance Heatmap\n(50% Overlay)', fontsize=11, fontweight='bold')
    axes[1, 2].axis('off')

    axes[1, 3].imshow(np.concatenate([original_np, highlighted], axis=1))
    axes[1, 3].set_title('Before | After\n(HFPN Score-CAM)', fontsize=11, fontweight='bold')
    axes[1, 3].axis('off')

    plt.tight_layout()
    plt.show()


def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss',   linewidth=2)
    ax1.plot(epochs, val_losses,   'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12); ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11); ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, train_accuracies, 'b-', label='Training Accuracy',   linewidth=2)
    ax2.plot(epochs, val_accuracies,   'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12); ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


def plot_confusion_matrix(cm, classes, metrics):
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=ax,
                annot_kws={'size': 16})
    ax.set_xlabel('Predicted Label', fontsize=13, fontweight='bold')
    ax.set_ylabel('True Label',      fontsize=13, fontweight='bold')
    ax.set_title(
        f'Confusion Matrix — Chest X-Ray Pneumonia CNN\n'
        f'Accuracy: {metrics["accuracy"]*100:.2f}%  |  '
        f'F1: {metrics["f1"]*100:.2f}%  |  '
        f'Kappa: {metrics["kappa"]:.4f}',
        fontsize=12, fontweight='bold'
    )
    plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
    plt.tight_layout(); plt.show()


# ================== Step 10: Main Pipeline ==================

def main():
    print("=" * 80)
    print("Chest X-Ray Pneumonia — Hierarchical FPN Frequency CNN")
    print("Dataset: Chest X-Ray Images (Pneumonia) [NORMAL / PNEUMONIA]")
    print("Architecture: ResNet50 + FreqAttentionGate(L2,L3) + DualDomain(L4) + FPN")
    print("Metrics: Accuracy, Precision, Recall, F1, Cohen Kappa, Specificity")
    print("=" * 80)

    # ---------- Update this path to your Kaggle input location ----------
    data_root = os.path.expanduser(
        '/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray'
    )
    # --------------------------------------------------------------------

    if not os.path.exists(data_root):
        print(f"\nERROR: Dataset path not found: {data_root}")
        print("Expected structure:")
        print("  data_root/train/NORMAL/   data_root/train/PNEUMONIA/")
        print("  data_root/val/NORMAL/     data_root/val/PNEUMONIA/")
        print("  data_root/test/NORMAL/    data_root/test/PNEUMONIA/")
        return

    print("\n[Step 1] Loading Chest X-Ray dataset...")
    try:
        trainset, valset, testset, classes = load_chest_xray_dataset(data_root)
    except Exception as e:
        print(f"ERROR loading dataset: {e}"); return

    print(f"\nClasses: {classes}")
    print(f"Train: {len(trainset)} | Val: {len(valset)} | Test: {len(testset)}")

    # The val split in this dataset is tiny (16 images).
    # Optionally, augment val with a small slice of the training set:
    val_count = len(valset)
    if val_count < 100:
        print(f"\n[Note] Val set has only {val_count} images — augmenting with 10% of train.")
        extra_size = int(0.10 * len(trainset))
        keep_size  = len(trainset) - extra_size
        keep_sub, extra_sub = torch.utils.data.random_split(
            trainset, [keep_size, extra_size],
            generator=torch.Generator().manual_seed(42)
        )
        # Rebuild loaders with augmented val
        trainset_used = keep_sub
        valset_used   = torch.utils.data.ConcatDataset([valset, extra_sub])
        print(f"  Adjusted Train: {len(trainset_used)} | Adjusted Val: {len(valset_used)}")
    else:
        trainset_used = trainset
        valset_used   = valset

    print("\n[Step 2] Converting to frequency domain...")
    freq_train = FrequencyDomainDataset(trainset_used, cache_freq=False)
    freq_val   = FrequencyDomainDataset(valset_used,   cache_freq=False)
    freq_test  = FrequencyDomainDataset(testset,       cache_freq=False)

    # Weighted sampler for class imbalance in training set
    if hasattr(trainset_used, 'dataset'):
        # It's a Subset — gather labels manually
        base_labels = [trainset_used.dataset.samples[i][1] for i in trainset_used.indices]
    elif hasattr(trainset_used, 'samples'):
        base_labels = [s[1] for s in trainset_used.samples]
    else:
        base_labels = [trainset_used[i][1] for i in range(len(trainset_used))]
    label_counts = Counter(base_labels)
    total = len(base_labels)
    sample_weights = [total / (len(label_counts) * label_counts[lbl]) for lbl in base_labels]
    sampler = WeightedRandomSampler(
        weights=sample_weights, num_samples=len(sample_weights), replacement=True
    )

    # Class weights for loss function (same inverse-frequency logic)
    class_weights = torch.FloatTensor([
        total / (len(label_counts) * label_counts[c]) for c in range(len(classes))
    ])
    print(f"Class weights for loss: {dict(zip(classes, class_weights.tolist()))}")

    batch_size  = 64
    num_workers = 4 if os.name != 'nt' else 0
    dl_kwargs   = dict(pin_memory=True, persistent_workers=False,
                       prefetch_factor=2 if num_workers > 0 else None)
    train_loader = DataLoader(freq_train, batch_size=batch_size, sampler=sampler,
                              num_workers=num_workers, **dl_kwargs)
    val_loader   = DataLoader(freq_val,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, **dl_kwargs)
    test_loader  = DataLoader(freq_test,  batch_size=1, shuffle=False, num_workers=0)

    print("\n[Step 3] Initialising Chest X-Ray Frequency-Domain CNN...")
    model = ChestXRayFrequencyCNN(num_classes=len(classes), dropout_rate=0.5).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

    print("\n[Step 4] Training...")
    try:
        train_losses, val_losses, train_accuracies, val_accuracies = train_model(
            model, train_loader, val_loader, class_weights,
            epochs=100, lr=0.001, weight_decay=5e-4
        )
    except Exception as e:
        import traceback
        print(f"\nERROR during training: {e}")
        traceback.print_exc(); return

    print("\n[Step 4.1] Training curves...")
    plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies)

    print("\n[Step 5] Test set evaluation...")
    model.eval()
    all_predictions, all_labels = [], []
    with torch.no_grad():
        for freq_images, labels, _ in tqdm(test_loader, desc="Testing"):
            freq_images, labels = freq_images.to(device), labels.to(device)
            outputs = model(freq_images)
            _, predicted = torch.max(outputs.data, 1)
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\n[Step 5.1] Computing comprehensive metrics...")
    metrics = compute_all_metrics(all_labels, all_predictions, classes)

    print("\n[Step 5.2] Confusion matrix...")
    plot_confusion_matrix(metrics['confusion_matrix'], classes, metrics)

    print("\n[Step 6] Score-CAM visualisations (5 random test samples)...")
    np.random.seed(42)
    test_indices = np.random.choice(len(testset), min(5, len(testset)), replace=False)
    for idx in test_indices:
        try:
            original_image, true_label = testset[idx]
            freq_features, phase, _ = spatial_to_frequency(original_image.unsqueeze(0))
            freq_input = freq_features.to(device)
            model.eval()
            with torch.no_grad():
                output = model(freq_input)
                probs  = F.softmax(output, dim=1)
                confidence, predicted = torch.max(probs.data, 1)
                predicted_class = predicted.item()
                confidence      = confidence.item() * 100
            print(f"Sample {idx}: True={classes[true_label]}, "
                  f"Pred={classes[predicted_class]} ({confidence:.1f}%)")
            _, saliency_map, highlighted, original_np, cam_freq = apply_scorecam_and_map_to_spatial(
                model, freq_input, phase.squeeze(0), predicted_class, original_image
            )
            plot_scorecam_results(original_np, freq_features, cam_freq, saliency_map,
                                  highlighted, predicted_class, true_label, classes, confidence)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"ERROR processing sample {idx}: {e}")

    print("\n[Step 7] Saving model...")
    try:
        torch.save({
            'model_state_dict': model.state_dict(),
            'test_accuracy':    metrics['accuracy'],
            'test_f1':          metrics['f1'],
            'test_kappa':       metrics['kappa'],
            'classes':          classes,
            'num_classes':      len(classes),
            'metrics':          metrics
        }, 'chest_xray_hfpn_cnn.pth')
        print("Saved as 'chest_xray_hfpn_cnn.pth'")
    except Exception as e:
        print(f"ERROR saving: {e}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("\n" + "=" * 80)
    print("PIPELINE COMPLETE — FINAL METRICS SUMMARY")
    print("=" * 80)
    print(f"  Overall Accuracy   : {metrics['accuracy']*100:.4f}%")
    print(f"  Overall Precision  : {metrics['precision']*100:.4f}%")
    print(f"  Overall Recall     : {metrics['recall']*100:.4f}%")
    print(f"  Overall F1 Score   : {metrics['f1']*100:.4f}%")
    print(f"  Overall Specificity: {metrics['specificity']*100:.4f}%")
    print(f"  Cohen's Kappa      : {metrics['kappa']:.4f}")
    print("=" * 80)
    print("\nKEY ARCHITECTURE NOTES:")
    print("  FrequencyAttentionGate @ layer2 (28x28) and layer3 (14x14)")
    print("  DualDomainFeatureFusion @ layer4 (7x7)")
    print("  FPN top-down merge with learnable softmax weights")
    print("  WeightedRandomSampler + Weighted CE loss for class imbalance")
    print("  Mild label smoothing (0.05) for binary task stability")
    print("=" * 80)


if __name__ == "__main__":
    main()